# Loan Management

## Import Libraries

In [1]:
from datetime import date, timedelta
import os
import sqlite3
import tkinter as tk
from tkinter import messagebox

import pandas as pd
import pandastable as pt
import ttkbootstrap as ttk

from ipynb.fs.full.base_frame import BaseFrame

## Loan Management Class

In [2]:
class MemoryVar:
    """Minimal variable-like object used in non-UI unit tests."""

    def __init__(self, value=""):
        self.value = value

    def get(self):
        return self.value

    def set(self, value):
        self.value = value


# noinspection PyTypeChecker
class LoanManagement(BaseFrame):
    """Manage loan operations, filtering, inventory, and SQLite persistence.

    This frame handles user/book selection, loan registration and returns,
    and synchronization of available stock using shared references provided
    by the DigitalLibrary controller.
    """
    def __init__(
        self,
        master=None,
        shared_categories=None,
        shared_users=None,
        shared_loans=None,
        shared_book_totals=None,
        no_ui=False
    ):
        """Initialize shared state references, UI variables, and table models.

        Args:
            master: Parent Tk container.
            shared_categories: Shared category and books mapping.
            shared_users: Shared user and type of users.
            shared_loans: Shared global loan list.
            shared_book_totals: Shared ISBN to total stock mapping.
            no_ui: If True, skip BaseFrame/UI creation for unit tests.
        """
        self.categories = shared_categories if shared_categories is not None else {}
        self.users = shared_users if shared_users is not None else {}
        self.loans = shared_loans if shared_loans is not None else []
        self.book_totals = shared_book_totals if shared_book_totals is not None else {}

        var_factory = tk.StringVar if not no_ui else MemoryVar
        self.user_name_filter_var = var_factory()
        self.user_email_filter_var = var_factory()
        self.book_title_filter_var = var_factory()
        self.book_isbn_filter_var = var_factory()
        self.book_category_filter_var = var_factory(value="All")
        self.book_author_filter_var = var_factory()

        self.selected_user_email = None
        self.selected_book_isbn = None
        self.selected_user_var = var_factory(value="Selected user: none")
        self.selected_book_var = var_factory(value="Selected book: none")

        self.user_df = pd.DataFrame(columns=["Name", "Email", "Type", "User ID"])
        self.book_df = pd.DataFrame(columns=["Title", "ISBN", "Category", "Author", "Available", "Total"])
        self.loan_df = pd.DataFrame(columns=["User Name", "User Email", "Book Title", "ISBN", "Loan Date", "Devolution Date", "Status"])
        self.loan_row_map = []

        self.user_table = None
        self.book_table = None
        self.loan_table = None

        if not no_ui:
            super().__init__(master)

    def get_title(self):
        """Return feature title shown by BaseFrame."""
        return "🏷️ Loan Management"

    def build_right_panel(self) -> None:
        """Build right-side action controls and selection summary labels."""
        super().build_right_panel()
        actions = ttk.Frame(master=self.right_panel, padding=(10, 8))
        actions.pack(fill=tk.X)

        ttk.Button(master=actions, text="Save Data", bootstyle="success", command=self.save_to_sqlite).pack(fill=tk.X, pady=(0, 8))
        ttk.Button(master=actions, text="Load Data", command=self.load_data).pack(fill=tk.X, pady=(0, 8))
        ttk.Separator(master=actions, orient=tk.HORIZONTAL).pack(fill=tk.X, pady=(14, 18))
        ttk.Button(master=actions, text="Confirm Loan", bootstyle="success", command=self.confirm_loan).pack(fill=tk.X, pady=(0, 8))
        ttk.Button(master=actions, text="Return Selected Loan", bootstyle="warning", command=self.return_selected_loan).pack(fill=tk.X, pady=(0, 8))
        ttk.Button(master=actions, text="Refresh Tables", command=self.refresh_all_tables).pack(fill=tk.X)
        ttk.Separator(master=actions, orient=tk.HORIZONTAL).pack(fill=tk.X, pady=(22, 18))
        ttk.Label(master=actions, textvariable=self.selected_user_var, style="normal.TLabel", wraplength=220).pack(fill=tk.X, pady=(0, 6))
        ttk.Label(master=actions, textvariable=self.selected_book_var, style="normal.TLabel", wraplength=220).pack(fill=tk.X)

    def build_left_panel(self) -> None:
        """Build users/books/loans sections and trigger initial refresh."""
        super().build_left_panel()
        self.left_panel_container.grid_columnconfigure(0, weight=1)
        self.left_panel_container.grid_columnconfigure(1, weight=0)
        self.left_panel_container.grid_columnconfigure(2, weight=1)
        self.left_panel_container.grid_rowconfigure(0, weight=1)
        self.left_panel_container.grid_rowconfigure(1, weight=1)
        self.left_panel_container.grid_rowconfigure(2, weight=1)
        ttk.Separator(master=self.left_panel_container, orient=tk.VERTICAL).grid(row=0, column=1, rowspan=2, sticky=tk.NS, padx=2, pady=(33, 10))
        self.build_user_section()
        self.build_book_section()
        self.build_loans_section()
        self.refresh_all_tables()

    def _build_section_header(self, master, title: str) -> None:
        """Render section title and horizontal separator."""
        ttk.Label(master=master, text=title, style="h9.TLabel").pack(fill=tk.X)
        sep = tk.Frame(master=master, height=1)
        sep.pack(fill=tk.X, pady=(0, 10))
        sep.configure(background="#444444")

    def build_user_section(self) -> None:
        """Build user filter form and user selection table."""
        section = tk.Frame(master=self.left_panel_container)
        section.grid(row=0, column=0, rowspan=2, sticky=tk.NSEW, padx=10, pady=(10, 10))
        self._build_section_header(section, "Users")
        frame = tk.LabelFrame(master=section, text="User Filters")
        frame.pack(fill=tk.X, pady=(0, 10))
        ttk.Label(master=frame, text="Name").grid(row=0, column=0, sticky=tk.W, padx=(10, 8), pady=(8, 8))
        ttk.Entry(master=frame, textvariable=self.user_name_filter_var).grid(row=0, column=1, sticky="ew", padx=(0, 8), pady=(8, 8))
        ttk.Label(master=frame, text="Email").grid(row=0, column=2, sticky=tk.W, padx=(2, 8), pady=(8, 8))
        ttk.Entry(master=frame, textvariable=self.user_email_filter_var).grid(row=0, column=3, sticky="ew", padx=(0, 8), pady=(8, 8))
        ttk.Button(master=frame, text="Apply User Filter", command=self.apply_user_filter).grid(row=0, column=4, sticky="e", padx=(0, 10), pady=(8, 8))
        frame.grid_rowconfigure(1, minsize=48)
        frame.grid_columnconfigure(1, weight=1)
        frame.grid_columnconfigure(3, weight=1)
        user_host = tk.Frame(master=section)
        user_host.pack(fill=tk.BOTH, expand=True)
        self.user_table = self._create_table(user_host, self.user_df, height=150)
        self.user_table.bind("<ButtonRelease-1>", lambda _event: self.select_user_from_table())

    def build_book_section(self) -> None:
        """Build book filter form and book selection table."""
        section = tk.Frame(master=self.left_panel_container)
        section.grid(row=0, column=2, rowspan=2, sticky=tk.NSEW, padx=10, pady=(10, 10))
        self._build_section_header(section, "Books")
        frame = tk.LabelFrame(master=section, text="Book Filters")
        frame.pack(fill=tk.X, pady=(0, 10))
        ttk.Label(master=frame, text="Title").grid(row=0, column=0, sticky=tk.W, padx=(10, 8), pady=(8, 8))
        ttk.Entry(master=frame, textvariable=self.book_title_filter_var).grid(row=0, column=1, sticky="ew", padx=(0, 8), pady=(8, 8))
        ttk.Label(master=frame, text="ISBN").grid(row=0, column=2, sticky=tk.W, padx=(2, 8), pady=(8, 8))
        ttk.Entry(master=frame, textvariable=self.book_isbn_filter_var).grid(row=0, column=3, sticky="ew", padx=(0, 8), pady=(8, 8))
        ttk.Label(master=frame, text="Category").grid(row=1, column=2, sticky=tk.W, padx=(2, 8), pady=(8, 8))
        ttk.Combobox(master=frame, textvariable=self.book_category_filter_var, values=["All"] + list(self.categories.keys()), state="readonly").grid(row=1, column=3, sticky="ew", padx=(0, 8), pady=(8, 8))
        ttk.Label(master=frame, text="Author").grid(row=1, column=0, sticky=tk.W, padx=(10, 8), pady=(0, 8))
        ttk.Entry(master=frame, textvariable=self.book_author_filter_var).grid(row=1, column=1, sticky="ew", padx=(0, 8), pady=(0, 8))
        ttk.Button(master=frame, text="Apply Book Filter", command=self.apply_book_filter).grid(row=1, column=4, sticky="e", padx=(0, 10), pady=(0, 8))
        frame.grid_columnconfigure(1, weight=1)
        frame.grid_columnconfigure(3, weight=1)
        frame.grid_columnconfigure(5, weight=1)
        book_host = tk.Frame(master=section)
        book_host.pack(fill=tk.BOTH, expand=True)
        self.book_table = self._create_table(book_host, self.book_df, height=150)
        self.book_table.bind("<ButtonRelease-1>", lambda _event: self.select_book_from_table())

    def build_loans_section(self) -> None:
        """Build loans table section."""
        loan_host = tk.Frame(master=self.left_panel_container)
        loan_host.grid(row=2, column=0, columnspan=3, sticky=tk.NSEW, padx=10, pady=(0, 10))
        self._build_section_header(loan_host, "Loans")
        self.loan_table = self._create_table(loan_host, self.loan_df, height=150)

    @staticmethod
    def _table_options() -> dict:
        """Return shared visual configuration for all pandastables."""
        return {
            "align": "w",
            "cellbackgr": "#222222",
            "cellwidth": 170,
            "floatprecision": 0,
            "thousandseparator": "",
            "grid_color": "#949494",
            "linewidth": 0,
            "rowheight": 28,
            "horizlines": True,
            "vertlines": True,
            "showindex": False,
            "multipleselectioncolor": "#345475",
            "boxoutlinecolor": "#222222",
            "textcolor": "white",
            "font": "Arial",
            "fontsize": 10,
            "fontstyle": "",
            "rowheaderbgcolor": "#345475",
            "rowheaderfgcolor": "white",
            "rowselectedcolor": "#345475",
            "colheaderbgcolor": "#373737",
            "colheadertextcolor": "white",
            "colselectedcolor": "#345475"
        }

    def _create_table(self, master, dataframe: pd.DataFrame, height: int = 20):
        """Create a configured read-only pandastable instance.

        Args:
            master: Parent widget for the table.
            dataframe: Initial DataFrame bound to the table.
            height: Table height passed to pandastable.
        """
        table_frame = tk.Frame(master=master, padx=1, pady=1)
        table_frame.pack(fill=tk.BOTH, expand=True)
        table = pt.Table(table_frame, dataframe=dataframe, showtoolbar=False, showstatusbar=True,
                         editable=False, enable_menus=False, height=height, width=300)
        pt.config.apply_options(self._table_options(), table)
        table.show()
        return table

    def _set_table_data(self, table, dataframe: pd.DataFrame) -> None:
        """Replace table DataFrame and redraw the widget.

        Ensures `Available` and `Total` columns are rendered as integers.
        """
        data = dataframe
        if len(data) > 0:
            for column_name in ["Available", "Total"]:
                if column_name in data.columns:
                    data[column_name] = pd.to_numeric(data[column_name], errors="coerce").fillna(0).astype(int)
        if len(data) == 0:
            data = pd.DataFrame([[None] * len(data.columns)], columns=data.columns)
        table.model.df = data
        table.redraw()

    def _iter_users(self):
        """Yield normalized user records from shared users mapping."""
        for users in self.users.values():
            for user in users:
                if not hasattr(user, "name") or not hasattr(user, "email"):
                    continue
                user_type = getattr(getattr(user, "user_type", ""), "label", "")
                yield {
                    "name": str(user.name),
                    "email": str(user.email).strip().lower(),
                    "type": str(user_type),
                    "user_id": str(getattr(user, "user_id", ""))
                }

    def _iter_books(self):
        """Yield normalized book records from shared categories mapping.

        Initializes missing `book_totals` entries from current availability.
        """
        for category_name, books in self.categories.items():
            for book in books:
                if not hasattr(book, "isbn") or not hasattr(book, "title"):
                    continue
                isbn = str(book.isbn).strip()
                available = int(getattr(book, "quantity", 0))
                if isbn not in self.book_totals:
                    self.book_totals[isbn] = available
                yield {
                    "title": str(book.title),
                    "isbn": isbn,
                    "category": str(category_name),
                    "author": str(getattr(book, "author", "")),
                    "available": available,
                    "total": int(self.book_totals.get(isbn, available))
                }

    def users_to_dataframe(self, name: str = "", email: str = "") -> pd.DataFrame:
        """Build users DataFrame using optional name/email substring filters."""
        name_filter = (name or "").strip().lower()
        email_filter = (email or "").strip().lower()
        rows = []
        for row in self._iter_users():
            if name_filter and name_filter not in row["name"].lower():
                continue
            if email_filter and email_filter not in row["email"].lower():
                continue
            rows.append(row)
        df = pd.DataFrame(rows, columns=["name", "email", "type", "user_id"])
        return df.rename(columns={"name": "Name", "email": "Email", "type": "Type", "user_id": "User ID"})

    def books_to_dataframe(self, title: str = "", isbn: str = "", category: str = "All", author: str = "") -> pd.DataFrame:
        """Build books DataFrame using title/isbn/category/author filters."""
        title_filter = (title or "").strip().lower()
        isbn_filter = (isbn or "").strip().lower()
        author_filter = (author or "").strip().lower()
        selected_category = (category or "All").strip()
        rows = []
        for row in self._iter_books():
            if selected_category.lower() != "all" and row["category"] != selected_category:
                continue
            if title_filter and title_filter not in row["title"].lower():
                continue
            if isbn_filter and isbn_filter not in row["isbn"].lower():
                continue
            if author_filter and author_filter not in row["author"].lower():
                continue
            rows.append(row)
        df = pd.DataFrame(rows, columns=["title", "isbn", "category", "author", "available", "total"])
        if len(df) > 0:
            df["available"] = pd.to_numeric(df["available"], errors="coerce").fillna(0).astype(int)
            df["total"] = pd.to_numeric(df["total"], errors="coerce").fillna(0).astype(int)
        return df.rename(columns={
            "title": "Title",
            "isbn": "ISBN",
            "category": "Category",
            "author": "Author",
            "available": "Available",
            "total": "Total"
        })

    @staticmethod
    def _format_date_br(value: str) -> str:
        """Format ISO-like date strings to Brazilian format DD/MM/YYYY."""
        raw = str(value or "").strip()
        if not raw:
            return ""
        date_part = raw.split("T", 1)[0]
        try:
            parsed = date.fromisoformat(date_part)
            return parsed.strftime("%d/%m/%Y")
        except ValueError:
            return raw

    def loans_to_dataframe(self) -> pd.DataFrame:
        """Build loans DataFrame and refresh row-to-loan mapping cache."""
        rows = []
        self.loan_row_map = []
        for loan in self.loans:
            loan_date_value = self._format_date_br(loan.get("loan_date", ""))
            due_date_value = self._format_date_br(loan.get("due_date", ""))
            status = "Returned" if bool(loan.get("returned", False)) else "Open"
            rows.append({
                "user_name": loan.get("user_name", ""),
                "user_email": loan.get("user_email", ""),
                "book_title": loan.get("book_title", ""),
                "isbn": loan.get("isbn", ""),
                "loan_date": loan_date_value,
                "due_date": due_date_value,
                "status": status
            })
            self.loan_row_map.append(loan)
        df = pd.DataFrame(rows, columns=["user_name", "user_email", "book_title", "isbn", "loan_date", "due_date", "status"])
        return df.rename(columns={
            "user_name": "User Name",
            "user_email": "User Email",
            "book_title": "Book Title",
            "isbn": "ISBN",
            "loan_date": "Loan Date",
            "due_date": "Devolution Date",
            "status": "Status"
        })

    def apply_user_filter(self) -> None:
        """Apply current user filters and refresh users table."""
        self.refresh_user_table()

    def apply_book_filter(self) -> None:
        """Apply current book filters and refresh books table."""
        self.refresh_book_table()

    def refresh_user_table(self) -> None:
        """Recompute users DataFrame and redraw users table."""
        self.user_df = self.users_to_dataframe(
            name=self.user_name_filter_var.get(),
            email=self.user_email_filter_var.get()
        )
        if self.user_table is not None:
            self._set_table_data(self.user_table, self.user_df)

    def refresh_book_table(self) -> None:
        """Recompute books DataFrame and redraw books table."""
        self.book_df = self.books_to_dataframe(
            title=self.book_title_filter_var.get(),
            isbn=self.book_isbn_filter_var.get(),
            category=self.book_category_filter_var.get(),
            author=self.book_author_filter_var.get()
        )
        if self.book_table is not None:
            self._set_table_data(self.book_table, self.book_df)

    def refresh_loan_table(self) -> None:
        """Recompute loans DataFrame and redraw loans table."""
        self.loan_df = self.loans_to_dataframe()
        if self.loan_table is not None:
            self._set_table_data(self.loan_table, self.loan_df)

    def refresh_all_tables(self) -> None:
        """Refresh users, books, and loans tables."""
        self.refresh_user_table()
        self.refresh_book_table()
        self.refresh_loan_table()

    def select_user_from_table(self) -> None:
        """Capture selected user (single click) and update selection state."""
        if self.user_table is None or len(self.user_df) == 0:
            return
        row_index = self.user_table.getSelectedRow()
        if row_index is None or row_index < 0 or row_index >= len(self.user_df):
            return
        row = self.user_df.iloc[row_index]
        email = "" if pd.isna(row["Email"]) else str(row["Email"]).strip().lower()
        name = "" if pd.isna(row["Name"]) else str(row["Name"]).strip()
        if not email:
            return
        self.selected_user_email = email
        self.selected_user_var.set(f"Selected user: {name} ({email})")

    def select_book_from_table(self) -> None:
        """Capture selected book (single click) and update selection state."""
        if self.book_table is None or len(self.book_df) == 0:
            return
        row_index = self.book_table.getSelectedRow()
        if row_index is None or row_index < 0 or row_index >= len(self.book_df):
            return
        row = self.book_df.iloc[row_index]
        isbn = "" if pd.isna(row["ISBN"]) else str(row["ISBN"]).strip()
        title = "" if pd.isna(row["Title"]) else str(row["Title"]).strip()
        if not isbn:
            return
        self.selected_book_isbn = isbn
        self.selected_book_var.set(f"Selected book: {title} ({isbn})")

    def _find_user_by_email(self, email: str):
        """Return user object by normalized email, or None if not found."""
        normalized = (email or "").strip().lower()
        for users in self.users.values():
            for user in users:
                current_email = str(getattr(user, "email", "")).strip().lower()
                if normalized and current_email == normalized:
                    return user
        return None

    def _find_book_by_isbn(self, isbn: str):
        """Return book object by normalized ISBN, or None if not found."""
        normalized = (isbn or "").strip()
        for books in self.categories.values():
            for book in books:
                if str(getattr(book, "isbn", "")).strip() == normalized:
                    return book
        return None

    def confirm_loan(self) -> None:
        """Validate selection and register a new loan.

        Decreases book availability, creates a loan with current date,
        and sets devolution date to 30 days after loan date.
        """
        if not self.selected_user_email:
            messagebox.showwarning("Selection Required", "Select one user in the users table.")
            return
        if not self.selected_book_isbn:
            messagebox.showwarning("Selection Required", "Select one book in the books table.")
            return

        user = self._find_user_by_email(self.selected_user_email)
        if user is None:
            messagebox.showerror("Loan Error", "Selected user was not found.")
            return

        book = self._find_book_by_isbn(self.selected_book_isbn)
        if book is None:
            messagebox.showerror("Loan Error", "Selected book was not found.")
            return

        available = int(getattr(book, "quantity", 0))
        if available <= 0:
            messagebox.showwarning("Unavailable Book", "No available quantity for this book.")
            return

        isbn = str(book.isbn).strip()
        if isbn not in self.book_totals:
            self.book_totals[isbn] = available

        book.quantity = available - 1

        loan_date = date.today()
        due_date = loan_date + timedelta(days=30)
        self.loans.append({
            "user_name": str(getattr(user, "name", "")),
            "user_email": str(getattr(user, "email", "")).strip().lower(),
            "book_title": str(getattr(book, "title", "")),
            "isbn": isbn,
            "loan_date": loan_date.isoformat(),
            "due_date": due_date.isoformat(),
            "returned": False
        })

        self.refresh_book_table()
        self.refresh_loan_table()
        messagebox.showinfo("Success", "Loan registered successfully.")

    def return_selected_loan(self) -> None:
        """Return selected loan and restore one unit of book availability."""
        if self.loan_table is None or len(self.loan_row_map) == 0:
            messagebox.showwarning("Selection Required", "No loans available for return.")
            return

        row_index = self.loan_table.getSelectedRow()
        if row_index is None or row_index < 0 or row_index >= len(self.loan_row_map):
            messagebox.showwarning("Selection Required", "Select one loan in the loans table.")
            return

        loan = self.loan_row_map[row_index]
        if bool(loan.get("returned", False)):
            messagebox.showwarning("Return", "Selected loan is already returned.")
            return

        isbn = str(loan.get("isbn", "")).strip()
        book = self._find_book_by_isbn(isbn)
        if book is None:
            messagebox.showerror("Return Error", "The related book was not found.")
            return

        available = int(getattr(book, "quantity", 0))
        max_total = int(self.book_totals.get(isbn, available))
        book.quantity = min(available + 1, max_total)

        loan["returned"] = True
        self.refresh_book_table()
        self.refresh_loan_table()
        messagebox.showinfo("Success", "Book returned successfully.")

    def save_to_sqlite(self, db_path: str = "library.db") -> None:
        """Persist in-memory loans to SQLite table `loans`.

        Uses full refresh strategy: deletes all rows then reinserts.
        """
        with sqlite3.connect(db_path) as conn:
            cursor = conn.cursor()
            cursor.execute("""
                CREATE TABLE IF NOT EXISTS loans (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    user_name TEXT NOT NULL,
                    user_email TEXT NOT NULL,
                    book_title TEXT NOT NULL,
                    isbn TEXT NOT NULL,
                    loan_date TEXT NOT NULL,
                    due_date TEXT NOT NULL,
                    returned INTEGER NOT NULL DEFAULT 0,
                    book_total INTEGER NOT NULL DEFAULT 0
                )
            """)
            cursor.execute("DELETE FROM loans")
            for loan in self.loans:
                isbn = str(loan.get("isbn", "")).strip()
                cursor.execute(
                    """
                    INSERT INTO loans (user_name, user_email, book_title, isbn, loan_date, due_date, returned, book_total)
                    VALUES (?, ?, ?, ?, ?, ?, ?, ?)
                    """,
                    (
                        loan.get("user_name", ""),
                        loan.get("user_email", ""),
                        loan.get("book_title", ""),
                        isbn,
                        loan.get("loan_date", ""),
                        loan.get("due_date", ""),
                        1 if bool(loan.get("returned", False)) else 0,
                        int(self.book_totals.get(isbn, 0))
                    )
                )
            conn.commit()

    def _reset_book_availability_to_totals(self) -> None:
        """Reset all book quantities to tracked total stock values."""
        for books in self.categories.values():
            for book in books:
                isbn = str(getattr(book, "isbn", "")).strip()
                if not isbn:
                    continue
                if isbn not in self.book_totals:
                    self.book_totals[isbn] = int(getattr(book, "quantity", 0))
                book.quantity = int(self.book_totals[isbn])

    def _apply_open_loans_to_books(self) -> None:
        """Apply open-loan counts to derive current available quantities."""
        open_counts = {}
        for loan in self.loans:
            if bool(loan.get("returned", False)):
                continue
            isbn = str(loan.get("isbn", "")).strip()
            if not isbn:
                continue
            open_counts[isbn] = open_counts.get(isbn, 0) + 1
        for books in self.categories.values():
            for book in books:
                isbn = str(getattr(book, "isbn", "")).strip()
                if not isbn:
                    continue
                total = int(self.book_totals.get(isbn, int(getattr(book, "quantity", 0))))
                book.quantity = max(total - int(open_counts.get(isbn, 0)), 0)

    def load_data(self, db_path: str = "library.db") -> None:
        """Load loans from SQLite and rebuild availability state.

        Preserves shared loans reference by mutating `self.loans` in-place.
        """
        proceed = messagebox.askokcancel(
            "Load Data",
            "Current loan data will be replaced during load. Do you want to continue?"
        )
        if not proceed:
            return

        self.loans.clear()
        if not os.path.exists(db_path):
            self._reset_book_availability_to_totals()
            self.refresh_all_tables()
            return

        with sqlite3.connect(db_path) as conn:
            cursor = conn.cursor()
            cursor.execute("SELECT name FROM sqlite_master WHERE type='table' AND name='loans'")
            if cursor.fetchone() is None:
                self._reset_book_availability_to_totals()
                self.refresh_all_tables()
                return
            cursor.execute(
                """
                SELECT user_name, user_email, book_title, isbn, loan_date, due_date, returned, book_total
                FROM loans
                ORDER BY id
                """
            )
            rows = cursor.fetchall()

        for user_name, user_email, book_title, isbn, loan_date, due_date, returned, book_total in rows:
            norm_isbn = str(isbn or "").strip()
            if norm_isbn:
                self.book_totals[norm_isbn] = max(int(self.book_totals.get(norm_isbn, 0)), int(book_total or 0))
            self.loans.append({
                "user_name": str(user_name or ""),
                "user_email": str(user_email or "").strip().lower(),
                "book_title": str(book_title or ""),
                "isbn": norm_isbn,
                "loan_date": str(loan_date or ""),
                "due_date": str(due_date or ""),
                "returned": bool(returned)
            })

        self._reset_book_availability_to_totals()
        self._apply_open_loans_to_books()
        self.refresh_all_tables()


## Loan Management Tests

In [4]:
import unittest
from datetime import date
from types import SimpleNamespace
from unittest.mock import patch


class _RowSelectTable:
    def __init__(self, index=0):
        self.index = index
        self.model = SimpleNamespace(df=None)

    def getSelectedRow(self):
        return self.index

    def redraw(self):
        return None


class LoanManagementTest(unittest.TestCase):
    def _build_user(self, name, email, label="Student", user_id="u1"):
        return SimpleNamespace(
            name=name,
            email=email,
            user_id=user_id,
            user_type=SimpleNamespace(label=label)
        )

    def _build_book(self, title, isbn, quantity, author="Author"):
        return SimpleNamespace(
            title=title,
            isbn=isbn,
            quantity=quantity,
            author=author
        )

    def test_users_to_dataframe_columns_and_filter(self):
        users = {
            "Student": [self._build_user("Ana", "ana@mail.com", "Student", "1")],
            "Professor": [self._build_user("Bruno", "bruno@mail.com", "Professor", "2")]
        }
        management = LoanManagement(shared_users=users, no_ui=True)
        df = management.users_to_dataframe(name="ana")
        self.assertEqual(list(df.columns), ["Name", "Email", "Type", "User ID"])
        self.assertEqual(len(df), 1)
        self.assertEqual(df.iloc[0]["Email"], "ana@mail.com")

    def test_books_to_dataframe_integer_columns(self):
        categories = {
            "Technology": [
                self._build_book("Clean Code", "9780132350884", 5, "Robert C. Martin")
            ]
        }
        management = LoanManagement(shared_categories=categories, no_ui=True)
        df = management.books_to_dataframe(author="martin")
        self.assertEqual(list(df.columns), ["Title", "ISBN", "Category", "Author", "Available", "Total"])
        self.assertEqual(int(df.iloc[0]["Available"]), 5)
        self.assertEqual(int(df.iloc[0]["Total"]), 5)

    def test_confirm_loan_creates_record_and_decreases_quantity(self):
        user = self._build_user("Ana", "ana@mail.com")
        book = self._build_book("Dune", "9780441172719", 2, "Frank Herbert")
        users = {"Student": [user]}
        categories = {"Fiction": [book]}
        loans = []
        totals = {}

        management = LoanManagement(
            shared_users=users,
            shared_categories=categories,
            shared_loans=loans,
            shared_book_totals=totals,
            no_ui=True
        )
        management.selected_user_email = "ana@mail.com"
        management.selected_book_isbn = "9780441172719"

        with patch("tkinter.messagebox.showinfo"), patch("tkinter.messagebox.showwarning"), patch("tkinter.messagebox.showerror"):
            management.confirm_loan()

        self.assertEqual(book.quantity, 1)
        self.assertEqual(len(loans), 1)
        self.assertFalse(loans[0]["returned"])
        loan_date = date.fromisoformat(loans[0]["loan_date"])
        due_date = date.fromisoformat(loans[0]["due_date"])
        self.assertEqual((due_date - loan_date).days, 30)

    def test_return_selected_loan_marks_returned_and_restores_stock(self):
        user = self._build_user("Ana", "ana@mail.com")
        book = self._build_book("Dune", "9780441172719", 2, "Frank Herbert")
        users = {"Student": [user]}
        categories = {"Fiction": [book]}
        loans = []
        totals = {}

        management = LoanManagement(
            shared_users=users,
            shared_categories=categories,
            shared_loans=loans,
            shared_book_totals=totals,
            no_ui=True
        )
        management.selected_user_email = "ana@mail.com"
        management.selected_book_isbn = "9780441172719"

        with patch("tkinter.messagebox.showinfo"), patch("tkinter.messagebox.showwarning"), patch("tkinter.messagebox.showerror"):
            management.confirm_loan()

        management.loan_row_map = loans
        management.loan_table = _RowSelectTable(0)

        with patch("tkinter.messagebox.showinfo"), patch("tkinter.messagebox.showwarning"), patch("tkinter.messagebox.showerror"):
            management.return_selected_loan()

        self.assertTrue(loans[0]["returned"])
        self.assertEqual(book.quantity, 2)


unittest.main(argv=[''], exit=False)


....
----------------------------------------------------------------------
Ran 4 tests in 0.015s

OK
